# Generador de nombres de dinosaurios — entrenamiento y evaluación

Este notebook solo orquesta: la lógica vive en `labdino` (`preprocessing`, `model`, `training`, `reporting`).

**Diseño experimental (fijado antes de entrenar):**
- Dos configuraciones que solo difieren en capacidad: A = GRU 1 capa / hidden 128; B = GRU 2 capas / hidden 256 con dropout 0.2 entre capas.
- Mismo split, semilla, batch size, optimizador (Adam) y learning rate.
- Máximo de épocas fijo con early stopping por paciencia sobre la pérdida de validación; se conserva el mejor checkpoint.
- **Criterio de elección: menor pérdida de validación.**

In [ ]:
from pathlib import Path

import torch

from labdino.model import CharRNN, ModelConfig
from labdino.preprocessing import Vocab, build_dataset, load_names
from labdino.reporting import save_curves, save_summary
from labdino.seed import set_seed
from labdino.training import TrainConfig, split_train_val, train_model

ROOT = Path.cwd().parent if Path.cwd().name == "name_generator" else Path.cwd()
EVIDENCE = ROOT / "evidence"
ARTIFACTS = ROOT / "name_generator" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)  # CPU, reproducible

In [ ]:
SEED = 42
names = load_names(ROOT / "data" / "dinos.csv")
vocab = Vocab.from_names(names)
X, Y, T = build_dataset(names, vocab)
train, val = split_train_val(X, Y, val_fraction=0.1, seed=SEED)
print(f"{len(names)} nombres | |V|={len(vocab)} | T={T} | train={train[0].size(0)} val={val[0].size(0)}")

In [ ]:
train_cfg = TrainConfig(batch_size=64, lr=3e-3, max_epochs=100, patience=10, seed=SEED)
model_cfgs = {
    "A": ModelConfig(len(vocab), cell="gru", embedding_dim=32, hidden_size=128, num_layers=1, dropout=0.0, pad_id=vocab.pad_id),
    "B": ModelConfig(len(vocab), cell="gru", embedding_dim=32, hidden_size=256, num_layers=2, dropout=0.2, pad_id=vocab.pad_id),
}
print(train_cfg)

In [ ]:
results, rows = {}, []
for name, mcfg in model_cfgs.items():
    set_seed(SEED)
    model = CharRNN(mcfg)
    res = train_model(model, train, val, vocab.pad_id, train_cfg)
    results[name] = res
    torch.save({"state_dict": res.best_state, "model_config": mcfg.to_dict(), "best_epoch": res.best_epoch},
               ARTIFACTS / f"checkpoint_{name}.pt")
    rows.append({
        "config": name, "architecture": mcfg.cell.upper(), "num_layers": mcfg.num_layers,
        "hidden_size": mcfg.hidden_size, "embedding_dim": mcfg.embedding_dim, "dropout": mcfg.dropout,
        "epochs_run": res.epochs_run, "max_epochs": train_cfg.max_epochs, "patience": train_cfg.patience,
        "loss": "CrossEntropyLoss(ignore_index=<PAD>)", "optimizer": "Adam", "lr": train_cfg.lr,
        "batch_size": train_cfg.batch_size, "seed": SEED,
        "n_params": sum(p.numel() for p in model.parameters()),
        "best_val_loss": res.best_val_loss, "best_epoch": res.best_epoch,
        "final_train_loss": res.train_losses[-1],
    })
    print(f"Config {name}: épocas={res.epochs_run} mejor época={res.best_epoch} mejor val={res.best_val_loss:.4f}")

In [ ]:
CRITERION = "menor pérdida de validación"
winner = min(rows, key=lambda r: r["best_val_loss"])["config"]
save_curves(results, EVIDENCE)
save_summary(rows, winner, CRITERION, EVIDENCE)
print("Ganadora:", winner)

In [ ]:
from IPython.display import Image, Markdown, display

display(Image(EVIDENCE / "loss_curves_comparison.png"))
display(Markdown((EVIDENCE / "training_summary.md").read_text(encoding="utf-8")))